1. Génération des données du problème.
a) Générez les données du problème. Une matrice X de taille n = 200 individus et p = 2n variables. Vous prendrez soin de centrer la matrice et de la normaliser de sorte que Pn i=1 X2 ij = 1. Un vecteur de paramètre wopt ∈ IRp dont k = 5 seulement sont non nulles. Un vecteur de réponses y = Xwopt + ε ∈ IRn où ε est un bruit Gaussien entrainant un rapport signal sur bruit de 2.

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cvx
import time
n = 200 # number of examples (you can try with n = 1000 and n = 5000)
p = 2*n # dimensionality of the problem
k = 5 # number of active variables
np.random.seed(0)
X = np.random.randn(n,p) # creating features and normalizing them
X = (X - np.mean(X,axis = 0))/np.std(X,axis = 0)
t = np.arange(0,p)/(p-1); # bulding the variance matrix !
S = np.zeros((p,p))
nn = 0.00001
for i in range(p):
    S[i,:] = np.exp(-(t-t[i])**2/nn);
X = X@(S**.5)
X = X/np.linalg.norm(X,axis=0)
ind = np.random.choice(p, k, replace=False) # generating optimal weights
weights = np.random.randn(k)
weights += 0.1+np.sign(weights) # to get large enough weight
wopt = np.zeros(p)
wopt[ind] = weights
rsnr = 2 # generating output by X@w + noise
z = X[:,ind]@weights
stdnoise = np.std(z)/rsnr
y = z + stdnoise*np.random.randn(n)


b) Vérifiez que les données ont bien les propriétés attendues.

In [28]:
print(f"Shape X = {np.shape(X)}")
print(f"Sum X² = {np.sum(X**2)}")
print(f"k = {np.count_nonzero(wopt)}")
print(ind)


Shape X = (200, 400)
Sum X² = 400.00000000000006
k = 5
[309 390  55  82 329]


c) Calculez l’erreur de généralisation "in sample" de la méthode des moindres carrés,

In [ ]:
b_ls = np.linalg.solve(X.T@X,X.T@y)
e_ls = np.sum((X@b_ls-z)**2)
print("Test error for the LS regression: {:0.4f}".format(e_ls))


Test error for the LS regression: 1.3616


d) Écrire une fonction Eval_coef(X,z,coeff), qui calcule l’erreur de généralisation "in
sample"


2. Différentes manières de résoudre le problème du Lasso
a) Écrire un programme CVX résolvant, pour λ = 10−3n.


In [45]:
lam = 0.001*n
b = cvx.Variable(p)

o = cvx.Minimize(0.5*cvx.sum_squares(X@b-y) + lam*cvx.norm(b, 1))
problem = cvx.Problem(o)
problem.solve(solver=cvx.SCS, eps=1e-5)



2.7093979658031495

b) Vérifiez que la solution obtenue est meilleure que celle des moindres carrés

In [38]:
b_ls = np.linalg.solve(X.T@X, X.T@y)
e_ls = np.sum((X@b_ls-z)**2)
np.set_printoptions(formatter={'float': '{: 0.3f}'.format})
print("In sample error for the LS regression {: 0.4f}".format(e_ls))

e_la = np.sum((X@b.value-z)**2)
print("In sample error for the Lasso {: 0.4f}".format(e_la))

In sample error for the LS regression  2.7231
In sample error for the Lasso  0.3323


<div style="background-color:#f0f8ff; color:#333333; padding:10px; border-radius:8px; font-family:Arial, sans-serif; line-height:1.6;">
  <p>
    The error in Lasso is smaller than the MSE method, so we can assume it is better for this exemple.
  </p>
</div>

c) Écrire un programme CVX résolvant la formulation suivant de Lasso, avec une valeur de t permettant d’obtenir les mêmes résultats que le problème précédent.

In [51]:
t = np.sum(np.abs(b.value))
b1 = cvx.Variable(p)

o1 = cvx.Minimize(0.5*cvx.sum_squares(X@b1-y))
c = [cvx.norm(b1, 1) <= t]
problem = cvx.Problem(o1, c)

problem.solve(solver=cvx.SCS, eps=1e-5)

e_la1 = np.sum((X@b1.value-z)**2)
print("In sample error for the Lasso {: 0.4f}".format(e_la1))

In sample error for the Lasso  0.3322


In [55]:
print(c[0].dual_value)
print(lam)

0.1997949072001126
0.2


<div style="background-color:#f0f8ff; color:#333333; padding:10px; border-radius:8px; font-family:Arial, sans-serif; line-height:1.6;">
  <p>
    We can see that the dual value calculated with this method is indeed equal to the lambda defined previously
  </p>
</div>

d) Écrire un programme CVX résolvant la formulation suivant de Lasso, avec une valeur
de ε permettant d’obtenir les mêmes résultats que le problème précédent.


In [56]:
eps = np.sum((X@b1.value-y)**2)
b2 = cvx.Variable(p)

o2 = cvx.Minimize(cvx.norm(b2,1))
c2 = [0.5*cvx.sum_squares(X@b2-y) <= eps]
problem = cvx.Problem(o2, c2)

problem.solve(solver=cvx.SCS, eps=1e-5)

e_la2 = np.sum((X@b2.value-z)**2)
print("In sample error for the Lasso {: 0.4f}".format(e_la2))

In sample error for the Lasso  2.5041
